# 02 - CSV Chunking Hands-On

This notebook is a full practical lab for CSV chunking in RAG.

You will learn and practice:

1. Loading real CSV datasets
2. Creating row-level chunks (high precision lookup)
3. Creating grouped-summary chunks (analytics/comparative questions)
4. Adding robust metadata for filtering and traceability
5. Checking token limits before embedding
6. Running simple retrieval sanity checks
7. Exporting chunks to JSONL for indexing

Datasets used:
- `../data/csv/employee_workforce_data.csv`
- `../data/csv/ecommerce_orders_support_data.csv`


In [1]:
# Cell 1: Environment and imports

from __future__ import annotations

from pathlib import Path
from typing import Any
import json
import hashlib

import pandas as pd
import tiktoken

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print('pandas   :', pd.__version__)
print('tiktoken :', tiktoken.__version__)
print('Notebook ready.')


pandas   : 3.0.1
tiktoken : 0.12.0
Notebook ready.


In [2]:
# Cell 2: Paths and file existence check

DATA_DIR = Path('../data/csv')
EMPLOYEE_CSV = DATA_DIR / 'employee_workforce_data.csv'
ECOMMERCE_CSV = DATA_DIR / 'ecommerce_orders_support_data.csv'

print('Data directory:', DATA_DIR.resolve())
print('employee_workforce_data.csv exists:', EMPLOYEE_CSV.exists())
print('ecommerce_orders_support_data.csv exists:', ECOMMERCE_CSV.exists())

assert EMPLOYEE_CSV.exists(), f'Missing file: {EMPLOYEE_CSV}'
assert ECOMMERCE_CSV.exists(), f'Missing file: {ECOMMERCE_CSV}'


Data directory: C:\Users\Lenovo\source\repos\RAG_Handson\Handson\01-Chunking\data\csv
employee_workforce_data.csv exists: True
ecommerce_orders_support_data.csv exists: True


In [3]:
# Cell 3: Load both datasets

df_emp = pd.read_csv(EMPLOYEE_CSV)
df_ecom = pd.read_csv(ECOMMERCE_CSV)

print('Employee dataset shape :', df_emp.shape)
print('Ecommerce dataset shape:', df_ecom.shape)

display(df_emp.head(3))
display(df_ecom.head(3))


Employee dataset shape : (30, 12)
Ecommerce dataset shape: (30, 16)


,employee_id,employee_name,department,role,location,employment_type,start_date,salary_usd,performance_rating,last_promotion_date,manager_id,primary_skills
0,E001,Ananya Mehta,Engineering,Senior Backend Engineer,Bengaluru,Full-Time,2019-06-10,148000,4.7,2024-04-15,M101,Python;FastAPI;PostgreSQL;AWS
1,E002,Rohit Verma,Engineering,Platform Engineer,Hyderabad,Full-Time,2020-02-24,132000,4.3,2023-12-20,M101,Kubernetes;Terraform;Linux;Go
2,E003,Sneha Iyer,Product,Product Manager,Mumbai,Full-Time,2018-09-03,141000,4.5,2024-01-10,M201,Roadmapping;Analytics;Stakeholder Management


,record_id,order_id,customer_segment,region,order_date,product_category,unit_price_usd,quantity,discount_pct,payment_method,delivery_days,returned,support_ticket_id,issue_type,resolution_hours,customer_satisfaction
0,R001,O-10001,Enterprise,North America,2025-01-03,Electronics,329.99,2,10,Credit Card,3,No,T-5001,Delivery Delay,18,4
1,R002,O-10002,SMB,Europe,2025-01-04,Home Appliances,149.50,1,0,PayPal,5,No,T-5002,Payment Failure,6,5
2,R003,O-10003,Consumer,India,2025-01-04,Fashion,39.99,3,15,UPI,4,Yes,T-5003,Wrong Size,22,3


## Why two chunk layers for CSV?

CSV data usually needs **two retrieval behaviors**:

- **Row chunks** for exact lookup questions
  - Example: "What is salary of employee E012?"
- **Grouped summary chunks** for analytic questions
  - Example: "Average salary by department?"

If you create only row chunks, analytics questions are weak.
If you create only summaries, exact lookup questions are weak.

So we build both.


In [ ]:
# Cell 4: Utility functions

enc = tiktoken.get_encoding('cl100k_base')

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def stable_chunk_id(text: str, prefix: str) -> str:
    h = hashlib.md5(text.encode('utf-8')).hexdigest()[:12]
    return f'{prefix}_{h}'

def print_chunk_preview(chunks: list[Document], n: int = 3) -> None:
    n = min(n, len(chunks))
    for i in range(n):
        c = chunks[i]
        print('-' * 80)
        print(f'Chunk {i+1}/{len(chunks)} | chars={len(c.page_content)} | tokens={count_tokens(c.page_content)}')
        print('metadata:', c.metadata)
        print(c.page_content[:320])
        if len(c.page_content) > 320:
            print('...')


In [5]:
# Cell 5: Create row-level chunks from a dataframe

def build_row_chunks(df: pd.DataFrame, source_name: str, id_column: str | None = None) -> list[Document]:
    chunks: list[Document] = []
    total = len(df)

    for row_idx, row in df.iterrows():
        lines = [f'{col}: {row[col]}' for col in df.columns]
        text = '\n'.join(lines)

        primary_id = str(row[id_column]) if id_column and id_column in df.columns else str(row_idx)

        meta = {
            'source': source_name,
            'chunk_type': 'row',
            'row_index': int(row_idx),
            'primary_id': primary_id,
            'total_rows': total,
        }
        meta['chunk_id'] = stable_chunk_id(text, prefix='row')
        meta['char_count'] = len(text)
        meta['token_count'] = count_tokens(text)

        chunks.append(Document(page_content=text, metadata=meta))

    return chunks

emp_row_chunks = build_row_chunks(df_emp, source_name='employee_workforce_data.csv', id_column='employee_id')
ecom_row_chunks = build_row_chunks(df_ecom, source_name='ecommerce_orders_support_data.csv', id_column='record_id')

print('Employee row chunks :', len(emp_row_chunks))
print('Ecommerce row chunks:', len(ecom_row_chunks))
print_chunk_preview(emp_row_chunks, n=2)


Employee row chunks : 30
Ecommerce row chunks: 30
--------------------------------------------------------------------------------
Chunk 1/30 | chars=307 | tokens=93
metadata: {'source': 'employee_workforce_data.csv', 'chunk_type': 'row', 'row_index': 0, 'primary_id': 'E001', 'total_rows': 30, 'chunk_id': 'row_774c5948fa07', 'char_count': 307, 'token_count': 93}
employee_id: E001
employee_name: Ananya Mehta
department: Engineering
role: Senior Backend Engineer
location: Bengaluru
employment_type: Full-Time
start_date: 2019-06-10
salary_usd: 148000
performance_rating: 4.7
last_promotion_date: 2024-04-15
manager_id: M101
primary_skills: Python;FastAPI;PostgreSQL;AWS
--------------------------------------------------------------------------------
Chunk 2/30 | chars=300 | tokens=91
metadata: {'source': 'employee_workforce_data.csv', 'chunk_type': 'row', 'row_index': 1, 'primary_id': 'E002', 'total_rows': 30, 'chunk_id': 'row_a9e2e4cee3e2', 'char_count': 300, 'token_count': 91}
employee_id:

In [6]:
# Cell 6: Grouped summary chunks (analytics layer)

def build_group_summary_chunks(
    df: pd.DataFrame,
    source_name: str,
    group_by: str,
    include_numeric_stats: bool = True,
) -> list[Document]:
    assert group_by in df.columns, f'Column not found: {group_by}'
    chunks: list[Document] = []

    grouped = df.groupby(group_by, dropna=False)
    total_groups = grouped.ngroups
    numeric_cols = list(df.select_dtypes(include='number').columns)

    for g_idx, (group_value, gdf) in enumerate(grouped):
        lines: list[str] = []
        lines.append(f'Source file: {source_name}')
        lines.append(f'Group by: {group_by}')
        lines.append(f'Group value: {group_value}')
        lines.append(f'Record count: {len(gdf)}')
        lines.append('')

        if include_numeric_stats and numeric_cols:
            lines.append('Numeric summary:')
            for col in numeric_cols:
                lines.append(f'- {col}: min={gdf[col].min()}, max={gdf[col].max()}, mean={gdf[col].mean():.2f}')
            lines.append('')

        # Add a compact sample of records for context
        lines.append('Sample rows (up to 3):')
        sample = gdf.head(3)
        for _, row in sample.iterrows():
            row_text = '; '.join([f'{c}={row[c]}' for c in gdf.columns])
            lines.append(f'- {row_text}')

        text = '\n'.join(lines)

        meta = {
            'source': source_name,
            'chunk_type': 'group_summary',
            'group_by': group_by,
            'group_value': str(group_value),
            'group_index': int(g_idx),
            'total_groups': int(total_groups),
            'record_count': int(len(gdf)),
        }
        meta['chunk_id'] = stable_chunk_id(text, prefix='group')
        meta['char_count'] = len(text)
        meta['token_count'] = count_tokens(text)

        chunks.append(Document(page_content=text, metadata=meta))

    return chunks

emp_group_by_dept = build_group_summary_chunks(df_emp, 'employee_workforce_data.csv', group_by='department')
emp_group_by_location = build_group_summary_chunks(df_emp, 'employee_workforce_data.csv', group_by='location')

ecom_group_by_segment = build_group_summary_chunks(df_ecom, 'ecommerce_orders_support_data.csv', group_by='customer_segment')
ecom_group_by_issue = build_group_summary_chunks(df_ecom, 'ecommerce_orders_support_data.csv', group_by='issue_type')

print('Employee groups (department):', len(emp_group_by_dept))
print('Employee groups (location)  :', len(emp_group_by_location))
print('Ecom groups (segment)       :', len(ecom_group_by_segment))
print('Ecom groups (issue_type)    :', len(ecom_group_by_issue))

print_chunk_preview(emp_group_by_dept, n=2)


Employee groups (department): 10
Employee groups (location)  : 9
Ecom groups (segment)       : 3
Ecom groups (issue_type)    : 30
--------------------------------------------------------------------------------
Chunk 1/10 | chars=837 | tokens=251
metadata: {'source': 'employee_workforce_data.csv', 'chunk_type': 'group_summary', 'group_by': 'department', 'group_value': 'Design', 'group_index': 0, 'total_groups': 10, 'record_count': 2, 'chunk_id': 'group_21221221f569', 'char_count': 837, 'token_count': 251}
Source file: employee_workforce_data.csv
Group by: department
Group value: Design
Record count: 2

Numeric summary:
- salary_usd: min=98000, max=112000, mean=105000.00
- performance_rating: min=4.1, max=4.3, mean=4.20

Sample rows (up to 3):
- employee_id=E004; employee_name=Karan Bhatia; department=Design; role=UX Des
...
--------------------------------------------------------------------------------
Chunk 2/10 | chars=1155 | tokens=345
metadata: {'source': 'employee_workforce_data.

In [7]:
# Cell 7: Optional - Split long grouped summaries into smaller chunks

summary_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=120,
    separators=['\n\n', '\n', '. ', ' ', '']
)

split_emp_group_chunks = summary_splitter.split_documents(emp_group_by_dept)
split_ecom_group_chunks = summary_splitter.split_documents(ecom_group_by_issue)

for i, c in enumerate(split_emp_group_chunks):
    c.metadata['post_split_index'] = i
for i, c in enumerate(split_ecom_group_chunks):
    c.metadata['post_split_index'] = i

print('Before split (emp groups):', len(emp_group_by_dept))
print('After split  (emp groups):', len(split_emp_group_chunks))
print('Before split (ecom issues):', len(ecom_group_by_issue))
print('After split  (ecom issues):', len(split_ecom_group_chunks))


Before split (emp groups): 10
After split  (emp groups): 18
Before split (ecom issues): 30
After split  (ecom issues): 30


In [8]:
# Cell 8: Build final training chunk sets

employee_chunks = emp_row_chunks + emp_group_by_dept + emp_group_by_location
ecommerce_chunks = ecom_row_chunks + ecom_group_by_segment + ecom_group_by_issue
all_csv_chunks = employee_chunks + ecommerce_chunks

print('Employee total chunks :', len(employee_chunks))
print('Ecommerce total chunks:', len(ecommerce_chunks))
print('All CSV chunks        :', len(all_csv_chunks))

chunk_type_counts = pd.Series([c.metadata['chunk_type'] for c in all_csv_chunks]).value_counts()
display(chunk_type_counts.to_frame('count'))


Employee total chunks : 49
Ecommerce total chunks: 63
All CSV chunks        : 112


,count
row,60
group_summary,52


In [9]:
# Cell 9: Token safety checks

def token_stats(chunks: list[Document], model_limit: int = 256) -> pd.DataFrame:
    rows = []
    for c in chunks:
        t = count_tokens(c.page_content)
        rows.append({
            'source': c.metadata.get('source'),
            'chunk_type': c.metadata.get('chunk_type'),
            'char_count': len(c.page_content),
            'token_count': t,
            'within_limit': t <= model_limit,
        })
    return pd.DataFrame(rows)

stats_df = token_stats(all_csv_chunks, model_limit=256)
display(stats_df.describe(include='all'))

violations = stats_df[~stats_df['within_limit']]
print('Chunks above 256 tokens:', len(violations))
if not violations.empty:
    display(violations.head(10))
else:
    print('All chunks are within 256 token limit.')


,source,chunk_type,char_count,token_count,within_limit
count,112,112,112.000000,112.000000,112
unique,2,2,NaN,NaN,2
top,ecommerce_orders_support_data.csv,row,NaN,NaN,True
freq,63,60,NaN,NaN,96
mean,NaN,NaN,576.535714,180.357143,NaN
std,NaN,NaN,314.927102,97.834099,NaN
min,NaN,NaN,284.000000,86.000000,NaN
25%,NaN,NaN,313.500000,93.000000,NaN
50%,NaN,NaN,346.000000,106.000000,NaN
75%,NaN,NaN,765.500000,251.000000,NaN


Chunks above 256 tokens: 16


,source,chunk_type,char_count,token_count,within_limit
31,employee_workforce_data.csv,group_summary,1155,345,False
36,employee_workforce_data.csv,group_summary,869,263,False
37,employee_workforce_data.csv,group_summary,1149,338,False
38,employee_workforce_data.csv,group_summary,1148,343,False
39,employee_workforce_data.csv,group_summary,1188,345,False
40,employee_workforce_data.csv,group_summary,1148,348,False
41,employee_workforce_data.csv,group_summary,870,257,False
42,employee_workforce_data.csv,group_summary,1133,340,False
44,employee_workforce_data.csv,group_summary,1168,345,False
45,employee_workforce_data.csv,group_summary,849,257,False


In [10]:
# Cell 10: Lightweight retrieval sanity checks (keyword contains)

def keyword_search(chunks: list[Document], query_terms: list[str], top_n: int = 5) -> list[Document]:
    scored = []
    q_terms = [t.lower() for t in query_terms]

    for c in chunks:
        text = c.page_content.lower()
        score = sum(1 for term in q_terms if term in text)
        if score > 0:
            scored.append((score, c))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in scored[:top_n]]

test_queries = [
    ['engineering', 'average', 'salary'],
    ['contract', 'employee_id', 'E016'],
    ['issue_type', 'Defective Item'],
    ['customer_segment', 'Enterprise', 'discount_pct'],
]

for q in test_queries:
    print('\n' + '=' * 90)
    print('Query terms:', q)
    matches = keyword_search(all_csv_chunks, q, top_n=3)
    print('Matches:', len(matches))
    print_chunk_preview(matches, n=min(3, len(matches)))



Query terms: ['engineering', 'average', 'salary']
Matches: 3
--------------------------------------------------------------------------------
Chunk 1/3 | chars=307 | tokens=93
metadata: {'source': 'employee_workforce_data.csv', 'chunk_type': 'row', 'row_index': 0, 'primary_id': 'E001', 'total_rows': 30, 'chunk_id': 'row_774c5948fa07', 'char_count': 307, 'token_count': 93}
employee_id: E001
employee_name: Ananya Mehta
department: Engineering
role: Senior Backend Engineer
location: Bengaluru
employment_type: Full-Time
start_date: 2019-06-10
salary_usd: 148000
performance_rating: 4.7
last_promotion_date: 2024-04-15
manager_id: M101
primary_skills: Python;FastAPI;PostgreSQL;AWS
--------------------------------------------------------------------------------
Chunk 2/3 | chars=300 | tokens=91
metadata: {'source': 'employee_workforce_data.csv', 'chunk_type': 'row', 'row_index': 1, 'primary_id': 'E002', 'total_rows': 30, 'chunk_id': 'row_a9e2e4cee3e2', 'char_count': 300, 'token_count': 91}
em

In [11]:
# Cell 11: Save chunks as JSONL artifact (ready for vector indexing later)

OUTPUT_DIR = Path('../data/csv/chunk_artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

employee_out = OUTPUT_DIR / 'employee_chunks.jsonl'
ecom_out = OUTPUT_DIR / 'ecommerce_chunks.jsonl'
all_out = OUTPUT_DIR / 'all_csv_chunks.jsonl'

def save_jsonl(chunks: list[Document], path: Path) -> None:
    with path.open('w', encoding='utf-8') as f:
        for c in chunks:
            rec = {'page_content': c.page_content, 'metadata': c.metadata}
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

save_jsonl(employee_chunks, employee_out)
save_jsonl(ecommerce_chunks, ecom_out)
save_jsonl(all_csv_chunks, all_out)

print('Saved:')
print('-', employee_out.resolve())
print('-', ecom_out.resolve())
print('-', all_out.resolve())


Saved:
- C:\Users\Lenovo\source\repos\RAG_Handson\Handson\01-Chunking\data\csv\chunk_artifacts\employee_chunks.jsonl
- C:\Users\Lenovo\source\repos\RAG_Handson\Handson\01-Chunking\data\csv\chunk_artifacts\ecommerce_chunks.jsonl
- C:\Users\Lenovo\source\repos\RAG_Handson\Handson\01-Chunking\data\csv\chunk_artifacts\all_csv_chunks.jsonl


## Practice Tasks (Do these yourself)

1. Build `group_summary` chunks by `employment_type` for employee dataset and compare with `department` grouping.
2. Add a `high_salary_band` field (for example, `salary_usd >= 130000`) and generate summaries by this band.
3. For ecommerce dataset, add grouped summaries by `region` and `returned`.
4. Add one custom metadata field: `dataset_name` and verify it appears in all chunks.
5. Reduce chunk size for summary splitter from 900 to 500 and compare token stats.
6. Add a simple dedup check for `chunk_id` collisions.

Bonus:
- Build a tiny FAISS/Chroma index from `all_csv_chunks` and test semantic retrieval.


In [12]:
# Cell 12: Final summary

print('=' * 70)
print('CSV CHUNKING NOTEBOOK SUMMARY')
print('=' * 70)
print('Employee rows            :', len(df_emp))
print('Ecommerce rows           :', len(df_ecom))
print('Employee row chunks      :', len(emp_row_chunks))
print('Employee group chunks    :', len(emp_group_by_dept) + len(emp_group_by_location))
print('Ecommerce row chunks     :', len(ecom_row_chunks))
print('Ecommerce group chunks   :', len(ecom_group_by_segment) + len(ecom_group_by_issue))
print('Total chunks             :', len(all_csv_chunks))

stats = token_stats(all_csv_chunks, model_limit=256)
print('Max tokens in any chunk  :', int(stats['token_count'].max()))
print('Avg tokens per chunk     :', round(float(stats['token_count'].mean()), 1))
print('Chunks > 256 tokens      :', int((~stats['within_limit']).sum()))
print('\nNotebook complete. You are ready to index CSV chunks in next steps.')


CSV CHUNKING NOTEBOOK SUMMARY
Employee rows            : 30
Ecommerce rows           : 30
Employee row chunks      : 30
Employee group chunks    : 19
Ecommerce row chunks     : 30
Ecommerce group chunks   : 33
Total chunks             : 112
Max tokens in any chunk  : 446
Avg tokens per chunk     : 180.4
Chunks > 256 tokens      : 16

Notebook complete. You are ready to index CSV chunks in next steps.
